In [15]:
import pandas as pd
import numpy as np
from itertools import product, combinations


# Load data
df = pd.read_csv(r'../data/05_train_with_category.csv')

print("="*70)
print("Sample count by category")
print("="*70)
print(df['Category'].value_counts())

# Rule configuration
rule_config = {
    'Brute Force': {'Bwd Pkts/s': 'high', 'Init Fwd Win Byts': 'high'},
    'Botnet': {'Flow Duration': 'low', 'Pkt Len Mean': 'low'},
    'DoS/DDoS': {'PSH Flag Cnt': 'high', 'Init Fwd Win Byts': 'high'},
    'Web Attack': {'Subflow Fwd Byts': 'high', 'Fwd Header Len': 'high', 'Fwd Pkt Len Std': 'high'}
}

# FPR limit per category (α)
fpr_limits = {
    'Brute Force': 0.01,   # Strict (already performing well)
    'Botnet': 0.05,        # Relaxed (currently 21% → targeting 5%)
    'DoS/DDoS': 0.05,      # Relaxed (currently 21% → targeting 5%)
    'Web Attack': 0.01     # Strict (already performing well)
}

# Per-experiment configuration
experiment_config = {
    'Web Attack': {
        'mode': 'k-of-n',
        'k': 2,  # At least 2 of 3 conditions satisfied
        'alpha': 0.01
    },
    'DoS/DDoS': {
        'mode': 'alpha_sweep',
        'alpha': 0.10  # Relaxed from 5% to 10%
    },
    'Brute Force': {
        'mode': 'standard',
        'alpha': 0.01
    },
    'Botnet': {
        'mode': 'skip',  # Excluded from experiment (failure case for presentation)
        'alpha': 0.05
    }
}

# Expanded threshold quantile candidates (finer granularity)
quantiles_high = [0.90, 0.925, 0.95, 0.975, 0.99, 0.995, 0.999]  # Benign-based (direction=high)
quantiles_low = [0.10, 0.075, 0.05, 0.025, 0.01, 0.005, 0.001]   # Benign-based (direction=low)
attack_quantiles_high = [0.01, 0.05, 0.10, 0.15, 0.20, 0.25]     # Attack-based (direction=high)
attack_quantiles_low = [0.99, 0.95, 0.90, 0.85, 0.80, 0.75]      # Attack-based (direction=low)

Sample count by category
Category
Benign         11307
DoS/DDoS        8145
Brute Force     2324
Botnet          1327
Web Attack       743
Name: count, dtype: int64


In [16]:
# Step 1: Change to >= / <= + Add FPR return
# ============================================================================
def clean_feature_series(s, feature: str):
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if feature in ["Init Fwd Win Byts"]:
        s = s[s >= 0]
    return s

def get_threshold_candidates(df, category, feature, direction):
    """
    Generate expanded threshold candidates
    """
    benign = clean_feature_series(df[df['Category'] == 'Benign'][feature], feature)
    attack = clean_feature_series(df[df['Category'] == category][feature], feature)
    
    candidates = {}
    
    if direction == 'high':
        # Benign upper quantiles
        for q in quantiles_high:
            val = benign.quantile(q)
            candidates[f'benign_p{int(q*100)}'] = val
        # Attack lower quantiles
        for q in attack_quantiles_high:
            val = attack.quantile(q)
            candidates[f'attack_p{int(q*100)}'] = val
    else:  # low
        # Benign lower quantiles
        for q in quantiles_low:
            val = benign.quantile(q)
            candidates[f'benign_p{int(q*100)}'] = val
        # Attack upper quantiles
        for q in attack_quantiles_low:
            val = attack.quantile(q)
            candidates[f'attack_p{int(q*100)}'] = val
    
    # Remove NaN + Remove duplicates
    cleaned = {}
    seen_vals = set()
    for name, val in candidates.items():
        if val is None or (isinstance(val, float) and np.isnan(val)):
            continue
        key = round(float(val), 6)
        if key in seen_vals:
            continue
        seen_vals.add(key)
        cleaned[name] = float(val)
    
    return cleaned

def check_single_condition(val, threshold, direction):
    """Check single condition"""
    if pd.isna(val) or np.isinf(val):
        return False
    if direction == 'high':
        return val >= threshold
    else:
        return val <= threshold

def evaluate_and_rule(df, category, thresholds_dict):
    """
    Evaluate entire AND rule as a single classifier (One-vs-Benign)
    """
    attack_df = df[df['Category'] == category].copy()
    benign_df = df[df['Category'] == 'Benign'].copy()
    
    def check_and_rule(row):
        for feature, (threshold, direction) in thresholds_dict.items():
            val = row[feature]
            if pd.isna(val) or np.isinf(val):
                return False
            if feature == "Init Fwd Win Byts" and val < 0:
                return False
            
            if direction == 'high':
                if val < threshold:
                    return False
            else:
                if val > threshold:
                    return False
        return True
    
    attack_hits = attack_df.apply(check_and_rule, axis=1)
    benign_hits = benign_df.apply(check_and_rule, axis=1)
    
    tp = attack_hits.sum()
    fn = len(attack_df) - tp
    fp = benign_hits.sum()
    tn = len(benign_df) - fp
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    return {
        'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
        'Precision': precision, 'Recall': recall, 'F1': f1, 'FPR': fpr
    }
    
def check_single_condition(val, threshold, direction):
    """Check single condition"""
    if pd.isna(val) or np.isinf(val):
        return False
    if direction == 'high':
        return val >= threshold
    else:
        return val <= threshold


def evaluate_k_of_n_rule(df, category, thresholds_dict, k):
    """k-of-n rule 평가: n개 조건 중 k개 이상 만족하면 탐지"""
    attack_df = df[df['Category'] == category]
    benign_df = df[df['Category'] == 'Benign']
    
    def check_k_of_n(row):
        satisfied = 0
        for feature, (threshold, direction) in thresholds_dict.items():
            val = row[feature]
            if feature == "Init Fwd Win Byts" and val < 0:
                continue
            if check_single_condition(val, threshold, direction):
                satisfied += 1
        return satisfied >= k
    
    attack_hits = attack_df.apply(check_k_of_n, axis=1)
    benign_hits = benign_df.apply(check_k_of_n, axis=1)
    
    tp = attack_hits.sum()
    fn = len(attack_df) - tp
    fp = benign_hits.sum()
    tn = len(benign_df) - fp
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    return {
        'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
        'Precision': precision, 'Recall': recall, 'F1': f1, 'FPR': fpr
    }

def evaluate_threshold(df, category, feature, threshold, direction):
    """
    Evaluate a single feature threshold (One-vs-Benign)
    """
    attack_df = df[df['Category'] == category]
    benign_df = df[df['Category'] == 'Benign']
    
    def check(row):
        val = row[feature]
        if pd.isna(val) or np.isinf(val):
            return False
        if feature == "Init Fwd Win Byts" and val < 0:
            return False
        if direction == 'high':
            return val >= threshold
        else:
            return val <= threshold
    
    attack_hits = attack_df.apply(check, axis=1)
    benign_hits = benign_df.apply(check, axis=1)
    
    tp = attack_hits.sum()
    fn = len(attack_df) - tp
    fp = benign_hits.sum()
    tn = len(benign_df) - fp
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    return {
        'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
        'Precision': precision, 'Recall': recall, 'F1': f1, 'FPR': fpr
    }

In [21]:
# Threshold Analysis
# ============================================================================
all_results = []

for category, features in rule_config.items():
    print(f"\n{'='*70}")
    print(f"[{category}]")
    print("="*70)
    
    for feature, direction in features.items():
        benign = clean_feature_series(df[df['Category'] == 'Benign'][feature], feature)
        attack = clean_feature_series(df[df['Category'] == category][feature], feature)
        
        print(f"\n--- {feature} ({direction}) ---")
        print(f"Benign: mean={benign.mean():.4f}, median={benign.median():.4f}")
        print(f"Attack: mean={attack.mean():.4f}, median={attack.median():.4f}")
        
        # Threshold 후보
        if direction == 'high':
            candidates = {
                'benign_p95': benign.quantile(0.95),
                'benign_p99': benign.quantile(0.99),
                'attack_p5': attack.quantile(0.05),
                'attack_p25': attack.quantile(0.25)
            }
        else:
            candidates = {
                'benign_p5': benign.quantile(0.05),
                'benign_p1': benign.quantile(0.01),
                'attack_p95': attack.quantile(0.95),
                'attack_p75': attack.quantile(0.75)
            }
        
        # Clean candidates: remove NaN and duplicates
        cleaned = {}
        seen_vals = set()

        for name, val in candidates.items():
            if val is None or (isinstance(val, float) and np.isnan(val)):
                continue

            key = round(float(val), 6)
            if key in seen_vals:
                continue

            seen_vals.add(key)
            cleaned[name] = float(val)

        candidates = cleaned

        
        # Print with FPR column
        print(f"\n{'Threshold':<15} {'Value':<15} {'Precision':<10} {'Recall':<10} {'F1':<10} {'FPR':<10}")
        print("-"*70)
        
        best_f1, best_name, best_value = 0, None, None
        for name, value in candidates.items():
            result = evaluate_threshold(df, category, feature, value, direction)
            print(f"{name:<15} {value:<15.4f} {result['Precision']:<10.4f} {result['Recall']:<10.4f} {result['F1']:<10.4f} {result['FPR']:<10.4f}")
            if result['F1'] > best_f1:
                best_f1, best_name, best_value, best_result = result['F1'], name, value, result
        
        print(f"\n★ 추천: {best_name} = {best_value:.4f} (F1={best_f1:.4f}, FPR={best_result['FPR']:.4f})")
        all_results.append({
            'Category': category, 'Feature': feature, 'Direction': direction,
            'Threshold_Name': best_name, 'Threshold': best_value,
            'Precision': best_result['Precision'], 'Recall': best_result['Recall'],
            'F1': best_f1, 'FPR': best_result['FPR']
        })

# Results DataFrame
results_df = pd.DataFrame(all_results)


[Brute Force]

--- Bwd Pkts/s (high) ---
Benign: mean=4640.9303, median=0.0312
Attack: mean=598121.2635, median=500000.0000

Threshold       Value           Precision  Recall     F1         FPR       
----------------------------------------------------------------------
benign_p95      43478.2609      0.8012     0.9849     0.8836     0.0502    
benign_p99      55555.5556      0.9258     0.9337     0.9297     0.0154    
attack_p5       52631.5789      0.8881     0.9660     0.9254     0.0250    
attack_p25      500000.0000     0.9908     0.7844     0.8756     0.0015    

★ 추천: benign_p99 = 55555.5556 (F1=0.9297, FPR=0.0154)

--- Init Fwd Win Byts (high) ---
Benign: mean=8277.8687, median=8192.0000
Attack: mean=26883.0000, median=26883.0000

Threshold       Value           Precision  Recall     F1         FPR       
----------------------------------------------------------------------
benign_p95      32818.1000      0.0000     0.0000     0.0000     0.0302    
benign_p99      65535.0000

In [18]:
# Final Rule Output
print("Final Threshold Summary")
print("="*70)
print(results_df[['Category', 'Feature', 'Direction', 'Threshold', 'Precision', 'Recall', 'F1', 'FPR']].to_string(index=False))

print("\n\nFinal Rules")
print("="*70)
for category in ['Brute Force', 'Botnet', 'DoS/DDoS', 'Web Attack']:
    cat_df = results_df[results_df['Category'] == category]
    print(f"\n[{category}]")
    print("IF")
    for i, (_, row) in enumerate(cat_df.iterrows()):
        op = ">=" if row['Direction'] == 'high' else "<="
        print(f"    {row['Feature']} {op} {row['Threshold']:.4f}")
        if i < len(cat_df) - 1:
            print("  AND")
    print(f"THEN \"{category}\"")

Final Threshold Summary
   Category           Feature Direction    Threshold  Precision   Recall       F1      FPR
Brute Force        Bwd Pkts/s      high 55555.555556   0.925768 0.933735 0.929734 0.015389
Brute Force Init Fwd Win Byts      high 26883.000000   0.818887 1.000000 0.900426 0.045459
     Botnet     Flow Duration       low 11204.000000   0.322829 0.949510 0.481836 0.233749
     Botnet      Pkt Len Mean       low    56.875000   0.169719 0.998493 0.290125 0.573273
   DoS/DDoS      PSH Flag Cnt      high     1.000000   0.615257 0.863475 0.718533 0.388963
   DoS/DDoS Init Fwd Win Byts      high  8192.000000   0.644934 0.863475 0.738373 0.342443
 Web Attack  Subflow Fwd Byts      high  3847.700000   0.629870 0.261104 0.369172 0.010082
 Web Attack    Fwd Header Len      high   811.280000   0.625000 0.255720 0.362942 0.010082
 Web Attack   Fwd Pkt Len Std      high   235.615693   0.365471 0.438762 0.398777 0.050057


Final Rules

[Brute Force]
IF
    Bwd Pkts/s >= 55555.5556
  AND

## Starting from the "Final AND Rule Performance Evaluation" function, append it so that the system performance table is immediately shown with the current thresholds

In [19]:
# Step 2: AND Rule Performance Evaluation (One-vs-Benign)
# ============================================================================
print("\n\n" + "="*70)
print("Step 2: AND Rule Combination Performance Evaluation (One-vs-Benign)")
print("="*70)

def evaluate_and_rule(df, category, thresholds_dict):
    """
    Evaluate the entire AND rule as a single classifier
    
    Parameters:
    - df: Full dataset
    - category: Attack category to evaluate
    - thresholds_dict: {feature: (threshold, direction), ...}
    
    Returns:
    - Dictionary of performance metrics
    
    Evaluation criteria:
    - TP: Rule hits among samples of the target category
    - FN: Rule misses among samples of the target category
    - FP: Rule hits among Benign samples (false positives)
    - TN: Rule misses among Benign samples
    """
    # Extract only the target category and Benign
    attack_df = df[df['Category'] == category].copy()
    benign_df = df[df['Category'] == 'Benign'].copy()
    
    # AND condition evaluation function
    def check_and_rule(row):
        for feature, (threshold, direction) in thresholds_dict.items():
            val = row[feature]
            # Handle NaN or Inf
            if pd.isna(val) or np.isinf(val):
                return False
            # Handle negative values for Init Fwd Win Byts
            if feature == "Init Fwd Win Byts" and val < 0:
                return False
            
            if direction == 'high':
                if val < threshold:  # Fails to satisfy >= threshold
                    return False
            else:  # low
                if val > threshold:  # Fails to satisfy <= threshold
                    return False
        return True  # All conditions satisfied
    
    # Evaluate rule for each sample
    attack_hits = attack_df.apply(check_and_rule, axis=1)
    benign_hits = benign_df.apply(check_and_rule, axis=1)
    
    tp = attack_hits.sum()
    fn = len(attack_df) - tp
    fp = benign_hits.sum()
    tn = len(benign_df) - fp
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    return {
        'Category': category,
        'TP': int(tp),
        'FP': int(fp),
        'FN': int(fn),
        'TN': int(tn),
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'FPR': fpr,
        'Attack_Total': len(attack_df),
        'Benign_Total': len(benign_df)
    }


# Extract current thresholds from results_df and evaluate AND rule
and_rule_results = []

for category in ['Brute Force', 'Botnet', 'DoS/DDoS', 'Web Attack']:
    cat_df = results_df[results_df['Category'] == category]
    
    # Build threshold dict: {feature: (threshold, direction)}
    thresholds_dict = {}
    for _, row in cat_df.iterrows():
        thresholds_dict[row['Feature']] = (row['Threshold'], row['Direction'])
    
    # Evaluate AND rule
    result = evaluate_and_rule(df, category, thresholds_dict)
    and_rule_results.append(result)
    
    # Detailed output
    print(f"\n[{category}]")
    print(f"  Rule conditions:")
    for feat, (thresh, direction) in thresholds_dict.items():
        op = ">=" if direction == 'high' else "<="
        print(f"    {feat} {op} {thresh:.4f}")
    print(f"\n  Performance:")
    print(f"    TP={result['TP']:,} / {result['Attack_Total']:,} (Recall={result['Recall']:.4f})")
    print(f"    FP={result['FP']:,} / {result['Benign_Total']:,} (FPR={result['FPR']:.4f})")
    print(f"    Precision={result['Precision']:.4f}, F1={result['F1']:.4f}")

# Overall summary table
and_results_df = pd.DataFrame(and_rule_results)

print("\n" + "="*70)
print("AND Rule Performance Summary Table")
print("="*70)
print(f"{'Category':<15} {'TP':>8} {'FP':>8} {'FN':>8} {'Precision':>10} {'Recall':>10} {'F1':>10} {'FPR':>10}")
print("-"*85)
for _, row in and_results_df.iterrows():
    print(f"{row['Category']:<15} {row['TP']:>8,} {row['FP']:>8,} {row['FN']:>8,} {row['Precision']:>10.4f} {row['Recall']:>10.4f} {row['F1']:>10.4f} {row['FPR']:>10.4f}")

# Diagnosis
print("\n" + "="*70)
print("Diagnosis: Root Cause Analysis of Recall Drop")
print("="*70)
for category in ['Brute Force', 'Botnet', 'DoS/DDoS', 'Web Attack']:
    cat_df = results_df[results_df['Category'] == category]
    and_result = and_results_df[and_results_df['Category'] == category].iloc[0]
    
    single_recalls = cat_df['Recall'].values
    and_recall = and_result['Recall']
    
    # Product of individual feature recalls (theoretical AND minimum)
    theoretical_min = np.prod(single_recalls)
    
    print(f"\n[{category}]")
    print(f"  Single Feature Recall: {', '.join([f'{r:.4f}' for r in single_recalls])}")
    print(f"  Theoretical minimum (product): {theoretical_min:.4f}")
    print(f"  Actual AND Recall: {and_recall:.4f}")
    
    if and_recall < min(single_recalls) * 0.8:
        print(f"  ⚠️ Warning: Recall drops significantly with AND combination!")



Step 2: AND Rule Combination Performance Evaluation (One-vs-Benign)

[Brute Force]
  Rule conditions:
    Bwd Pkts/s >= 55555.5556
    Init Fwd Win Byts >= 26883.0000

  Performance:
    TP=2,170 / 2,324 (Recall=0.9337)
    FP=59 / 11,307 (FPR=0.0052)
    Precision=0.9735, F1=0.9532

[Botnet]
  Rule conditions:
    Flow Duration <= 11204.0000
    Pkt Len Mean <= 56.8750

  Performance:
    TP=1,259 / 1,327 (Recall=0.9488)
    FP=2,461 / 11,307 (FPR=0.2177)
    Precision=0.3384, F1=0.4989

[DoS/DDoS]
  Rule conditions:
    PSH Flag Cnt >= 1.0000
    Init Fwd Win Byts >= 8192.0000

  Performance:
    TP=7,033 / 8,145 (Recall=0.8635)
    FP=3,775 / 11,307 (FPR=0.3339)
    Precision=0.6507, F1=0.7422

[Web Attack]
  Rule conditions:
    Subflow Fwd Byts >= 3847.7000
    Fwd Header Len >= 811.2800
    Fwd Pkt Len Std >= 235.6157

  Performance:
    TP=190 / 743 (Recall=0.2557)
    FP=17 / 11,307 (FPR=0.0015)
    Precision=0.9179, F1=0.4000

AND Rule Performance Summary Table
Category     

In [24]:
print("Step 3: FPR-Constrained Threshold Optimization")

optimized_results = {}

for category, features in rule_config.items():
    alpha = fpr_limits[category]
    print(f"\n{'='*70}")
    print(f"[{category}] FPR constraint: α ≤ {alpha}")
    print("="*70)
    
    # Generate candidates per feature
    feature_candidates = {}
    for feature, direction in features.items():
        candidates = get_threshold_candidates(df, category, feature, direction)
        feature_candidates[feature] = {
            'candidates': candidates,
            'direction': direction
        }
        print(f"\n{feature} ({direction}): {len(candidates)} candidates")
        # Print a few candidate values
        sorted_vals = sorted(candidates.items(), key=lambda x: x[1])
        for name, val in sorted_vals[:3]:
            print(f"  {name}: {val:.4f}")
        if len(sorted_vals) > 3:
            print(f"  ... ({len(sorted_vals)-3} more)")
    
    # Search all combinations
    feature_names = list(features.keys())
    all_candidate_lists = []
    for feat in feature_names:
        cands = feature_candidates[feat]['candidates']
        all_candidate_lists.append([(name, val) for name, val in cands.items()])
    
    # Count total combinations
    total_combinations = 1
    for lst in all_candidate_lists:
        total_combinations *= len(lst)
    print(f"\nTotal combinations: {total_combinations}")
    
    # Search
    valid_candidates = []  # Candidates satisfying FPR ≤ α
    
    for combo in product(*all_candidate_lists):
        # Build thresholds_dict
        thresholds_dict = {}
        combo_info = {}
        for i, feat in enumerate(feature_names):
            name, val = combo[i]
            direction = feature_candidates[feat]['direction']
            thresholds_dict[feat] = (val, direction)
            combo_info[feat] = {'name': name, 'value': val}
        
        # Evaluate AND rule
        result = evaluate_and_rule(df, category, thresholds_dict)
        
        # Constraint check: FPR ≤ α
        if result['FPR'] <= alpha:
            valid_candidates.append({
                'combo_info': combo_info,
                'thresholds_dict': thresholds_dict,
                'result': result
            })
    
    print(f"Candidates satisfying FPR ≤ {alpha}: {len(valid_candidates)}")
    
    if len(valid_candidates) == 0:
        print(f"Warning: No candidates satisfy the FPR constraint!")
        print(f"   → Consider raising α or revisiting features")
        optimized_results[category] = None
        continue
    
    # Objective: maximize Recall
    best = max(valid_candidates, key=lambda x: x['result']['Recall'])
    optimized_results[category] = best
    
    print(f"\n★ Optimal combination (max Recall):")
    for feat, info in best['combo_info'].items():
        direction = feature_candidates[feat]['direction']
        op = ">=" if direction == 'high' else "<="
        print(f"  {feat} {op} {info['value']:.4f} ({info['name']})")
    
    r = best['result']
    print(f"\n  Performance:")
    print(f"    Recall={r['Recall']:.4f}, FPR={r['FPR']:.4f}")
    print(f"    Precision={r['Precision']:.4f}, F1={r['F1']:.4f}")
    print(f"    TP={r['TP']}, FP={r['FP']}, FN={r['FN']}")


# ============================================================================
# Final Summary
# ============================================================================

print("\n\n" + "="*70)
print("Step 3 Final Summary: Optimized Rules")
print("="*70)

summary_rows = []

for category in ['Brute Force', 'Botnet', 'DoS/DDoS', 'Web Attack']:
    alpha = fpr_limits[category]
    best = optimized_results.get(category)
    
    print(f"\n[{category}] (α ≤ {alpha})")
    
    if best is None:
        print("No valid candidates")
        continue
    
    features_info = rule_config[category]
    print("  IF")
    conditions = []
    for feat, direction in features_info.items():
        val = best['combo_info'][feat]['value']
        op = ">=" if direction == 'high' else "<="
        print(f"      {feat} {op} {val:.4f}")
        conditions.append(f"{feat} {op} {val:.4f}")
    print(f"  THEN \"{category}\"")
    
    r = best['result']
    summary_rows.append({
        'Category': category,
        'FPR_Limit': alpha,
        'Recall': r['Recall'],
        'FPR': r['FPR'],
        'Precision': r['Precision'],
        'F1': r['F1'],
        'TP': r['TP'],
        'FP': r['FP']
    })

# Summary table
print("\n" + "="*70)
print("Performance Summary Table")
print("="*70)
summary_df = pd.DataFrame(summary_rows)
print(f"{'Category':<15} {'α':>6} {'Recall':>10} {'FPR':>10} {'Precision':>10} {'F1':>10}")
print("-"*65)
for _, row in summary_df.iterrows():
    print(f"{row['Category']:<15} {row['FPR_Limit']:>6.2f} {row['Recall']:>10.4f} {row['FPR']:>10.4f} {row['Precision']:>10.4f} {row['F1']:>10.4f}")


# ============================================================================
# Improvement Comparison vs Step 2
# ============================================================================

print("\n" + "="*70)
print("Improvement Comparison vs Step 2")
print("="*70)

# Step 2 results (from previous output)
step2_results = {
    'Brute Force': {'Recall': 0.9337, 'FPR': 0.0052, 'F1': 0.9532},
    'Botnet': {'Recall': 0.9488, 'FPR': 0.2177, 'F1': 0.4989},
    'DoS/DDoS': {'Recall': 0.8134, 'FPR': 0.2150, 'F1': 0.7703},
    'Web Attack': {'Recall': 0.2557, 'FPR': 0.0015, 'F1': 0.4000}
}

print(f"{'Category':<15} {'Step2 FPR':>12} {'Step3 FPR':>12} {'Step2 Recall':>14} {'Step3 Recall':>14}")
print("-"*70)
for category in ['Brute Force', 'Botnet', 'DoS/DDoS', 'Web Attack']:
    s2 = step2_results[category]
    best = optimized_results.get(category)
    if best:
        s3 = best['result']
        fpr_change = "↓" if s3['FPR'] < s2['FPR'] else ("↑" if s3['FPR'] > s2['FPR'] else "=")
        recall_change = "↓" if s3['Recall'] < s2['Recall'] else ("↑" if s3['Recall'] > s2['Recall'] else "=")
        print(f"{category:<15} {s2['FPR']:>10.4f}   {s3['FPR']:>10.4f} {fpr_change}  {s2['Recall']:>12.4f}   {s3['Recall']:>12.4f} {recall_change}")
    else:
        print(f"{category:<15} {s2['FPR']:>10.4f}   {'N/A':>10}    {s2['Recall']:>12.4f}   {'N/A':>12}")

Step 3: FPR-Constrained Threshold Optimization

[Brute Force] FPR constraint: α ≤ 0.01

Bwd Pkts/s (high): 9 candidates
  benign_p90: 621.2257
  benign_p92: 2808.9888
  attack_p1: 38815.3846
  ... (6 more)

Init Fwd Win Byts (high): 5 candidates
  benign_p90: 14600.0000
  benign_p92: 26883.0000
  benign_p95: 32818.1000
  ... (2 more)

Total combinations: 45
Candidates satisfying FPR ≤ 0.01: 27

★ Optimal combination (max Recall):
  Bwd Pkts/s >= 51754.3860 (benign_p97)
  Init Fwd Win Byts >= 26883.0000 (benign_p92)

  Performance:
    Recall=0.9660, FPR=0.0088
    Precision=0.9578, F1=0.9619
    TP=2245, FP=99, FN=79

[Botnet] FPR constraint: α ≤ 0.05

Flow Duration (low): 11 candidates
  benign_p1: 1.0000
  benign_p2: 18.0000
  benign_p5: 20.0000
  ... (8 more)

Pkt Len Mean (low): 2 candidates
  benign_p10: 0.0000
  attack_p99: 56.8750

Total combinations: 22
Candidates satisfying FPR ≤ 0.05: 5

★ Optimal combination (max Recall):
  Flow Duration <= 20.0000 (benign_p5)
  Pkt Len Mean

In [25]:
# Step 4A: Web Attack k-of-n (2-of-3)
# ============================================================================

print("\n" + "="*70)
print("Step 4A: Web Attack k-of-n (k=2, n=3)")
print("="*70)

category = 'Web Attack'
config = experiment_config[category]
features = rule_config[category]
k = config['k']
alpha = config['alpha']

print(f"Config: k={k} (at least {k} of 3 conditions satisfied)")
print(f"FPR constraint: α ≤ {alpha}")

# Generate candidates
feature_candidates = {}
for feature, direction in features.items():
    candidates = get_threshold_candidates(df, category, feature, direction)
    feature_candidates[feature] = {
        'candidates': candidates,
        'direction': direction
    }
    print(f"\n{feature} ({direction}): {len(candidates)} candidates")

# Search all combinations
feature_names = list(features.keys())
all_candidate_lists = []
for feat in feature_names:
    cands = feature_candidates[feat]['candidates']
    all_candidate_lists.append([(name, val) for name, val in cands.items()])

total_combinations = 1
for lst in all_candidate_lists:
    total_combinations *= len(lst)
print(f"\nTotal combinations: {total_combinations}")

# k-of-n search
valid_candidates = []

for combo in product(*all_candidate_lists):
    thresholds_dict = {}
    combo_info = {}
    for i, feat in enumerate(feature_names):
        name, val = combo[i]
        direction = feature_candidates[feat]['direction']
        thresholds_dict[feat] = (val, direction)
        combo_info[feat] = {'name': name, 'value': val}
    
    # k-of-n evaluation
    result = evaluate_k_of_n_rule(df, category, thresholds_dict, k)
    
    if result['FPR'] <= alpha:
        valid_candidates.append({
            'combo_info': combo_info,
            'thresholds_dict': thresholds_dict,
            'result': result
        })

print(f"Candidates satisfying FPR ≤ {alpha}: {len(valid_candidates)}")

if len(valid_candidates) > 0:
    # Maximize Recall
    best_web = max(valid_candidates, key=lambda x: x['result']['Recall'])
    
    print(f"\n★ Optimal combination (k={k}, max Recall):")
    for feat, info in best_web['combo_info'].items():
        direction = feature_candidates[feat]['direction']
        op = ">=" if direction == 'high' else "<="
        print(f"  {feat} {op} {info['value']:.4f} ({info['name']})")
    
    r = best_web['result']
    print(f"\n  Performance:")
    print(f"    Recall={r['Recall']:.4f}, FPR={r['FPR']:.4f}")
    print(f"    Precision={r['Precision']:.4f}, F1={r['F1']:.4f}")
    print(f"    TP={r['TP']}, FP={r['FP']}, FN={r['FN']}")
    
    # Comparison vs Step 3 AND rule
    print(f"\n  vs Step 3 (AND):")
    print(f"    Recall: 0.2638 → {r['Recall']:.4f} ({'+' if r['Recall'] > 0.2638 else ''}{(r['Recall']-0.2638)*100:.1f}%p)")
    print(f"    FPR:    0.0096 → {r['FPR']:.4f}")
else:
    best_web = None
    print("No valid candidates")

# ============================================================================
# Step 4B: DoS/DDoS alpha = 0.10
# ============================================================================

print("\n\n" + "="*70)
print("Step 4B: DoS/DDoS α = 0.10 (Relaxed)")
print("="*70)

category = 'DoS/DDoS'
config = experiment_config[category]
features = rule_config[category]
alpha = config['alpha']

print(f"FPR constraint: α ≤ {alpha} (relaxed from 0.05)")

# Generate candidates
feature_candidates = {}
for feature, direction in features.items():
    candidates = get_threshold_candidates(df, category, feature, direction)
    feature_candidates[feature] = {
        'candidates': candidates,
        'direction': direction
    }
    print(f"\n{feature} ({direction}): {len(candidates)} candidates")

# Search all combinations
feature_names = list(features.keys())
all_candidate_lists = []
for feat in feature_names:
    cands = feature_candidates[feat]['candidates']
    all_candidate_lists.append([(name, val) for name, val in cands.items()])

total_combinations = 1
for lst in all_candidate_lists:
    total_combinations *= len(lst)
print(f"\nTotal combinations: {total_combinations}")

valid_candidates = []

for combo in product(*all_candidate_lists):
    thresholds_dict = {}
    combo_info = {}
    for i, feat in enumerate(feature_names):
        name, val = combo[i]
        direction = feature_candidates[feat]['direction']
        thresholds_dict[feat] = (val, direction)
        combo_info[feat] = {'name': name, 'value': val}
    
    result = evaluate_and_rule(df, category, thresholds_dict)
    
    if result['FPR'] <= alpha:
        valid_candidates.append({
            'combo_info': combo_info,
            'thresholds_dict': thresholds_dict,
            'result': result
        })

print(f"Candidates satisfying FPR ≤ {alpha}: {len(valid_candidates)}")

if len(valid_candidates) > 0:
    best_dos = max(valid_candidates, key=lambda x: x['result']['Recall'])
    
    print(f"\n★ Optimal combination (max Recall):")
    for feat, info in best_dos['combo_info'].items():
        direction = feature_candidates[feat]['direction']
        op = ">=" if direction == 'high' else "<="
        print(f"  {feat} {op} {info['value']:.4f} ({info['name']})")
    
    r = best_dos['result']
    print(f"\n  Performance:")
    print(f"    Recall={r['Recall']:.4f}, FPR={r['FPR']:.4f}")
    print(f"    Precision={r['Precision']:.4f}, F1={r['F1']:.4f}")
    print(f"    TP={r['TP']}, FP={r['FP']}, FN={r['FN']}")
    
    # Comparison vs Step 3
    print(f"\n  vs Step 3 (α=0.05):")
    print(f"    Recall: 0.5650 → {r['Recall']:.4f} ({'+' if r['Recall'] > 0.5650 else ''}{(r['Recall']-0.5650)*100:.1f}%p)")
    print(f"    FPR:    0.0438 → {r['FPR']:.4f}")
else:
    best_dos = None
    print("No valid candidates")


Step 4A: Web Attack k-of-n (k=2, n=3)
Config: k=2 (at least 2 of 3 conditions satisfied)
FPR constraint: α ≤ 0.01

Subflow Fwd Byts (high): 3 candidates

Fwd Header Len (high): 7 candidates

Fwd Pkt Len Std (high): 6 candidates

Total combinations: 126
Candidates satisfying FPR ≤ 0.01: 22

★ Optimal combination (k=2, max Recall):
  Subflow Fwd Byts >= 24782.0440 (benign_p99)
  Fwd Header Len >= 400.0000 (benign_p95)
  Fwd Pkt Len Std >= 235.6157 (benign_p95)

  Performance:
    Recall=0.2611, FPR=0.0082
    Precision=0.6760, F1=0.3767
    TP=194, FP=93, FN=549

  vs Step 3 (AND):
    Recall: 0.2638 → 0.2611 (-0.3%p)
    FPR:    0.0096 → 0.0082


Step 4B: DoS/DDoS α = 0.10 (Relaxed)
FPR constraint: α ≤ 0.1 (relaxed from 0.05)

PSH Flag Cnt (high): 2 candidates

Init Fwd Win Byts (high): 6 candidates

Total combinations: 12
Candidates satisfying FPR ≤ 0.1: 10

★ Optimal combination (max Recall):
  PSH Flag Cnt >= 1.0000 (benign_p90)
  Init Fwd Win Byts >= 14600.0000 (benign_p90)

  Perf

In [26]:
# Step 4 Final Summary
# ============================================================================

print("\n\n" + "="*70)
print("Step 4 Final Summary")
print("="*70)

# Step 3 results (for comparison)
step3_results = {
    'Brute Force': {'Recall': 0.9660, 'FPR': 0.0088, 'Precision': 0.9578, 'F1': 0.9619},
    'Botnet': {'Recall': 0.0000, 'FPR': 0.0440, 'Precision': 0.0000, 'F1': 0.0000},
    'DoS/DDoS': {'Recall': 0.5650, 'FPR': 0.0438, 'Precision': 0.9029, 'F1': 0.6951},
    'Web Attack': {'Recall': 0.2638, 'FPR': 0.0096, 'Precision': 0.6426, 'F1': 0.3740}
}

print("\n[Final Rules by Category]")
print("-"*70)

# Brute Force (carry over from Step 3)
print("\n1. Brute Force (Step 3 unchanged, α=0.01)")
print("   IF Bwd Pkts/s >= 51754.3860")
print("      AND Init Fwd Win Byts >= 26883.0000")
print("   THEN \"Brute Force\"")
s3 = step3_results['Brute Force']
print(f"   → Recall={s3['Recall']:.4f}, FPR={s3['FPR']:.4f}, F1={s3['F1']:.4f}")

# Botnet (failure case)
print("\n2. Botnet (Step 3, α=0.05) Failed")
print("   IF Flow Duration <= 20.0000")
print("      AND Pkt Len Mean <= 0.0000")
print("   THEN \"Botnet\"")
print("   → Recall=0.0000, FPR=0.0440 (Detection not possible due to feature limitations)")

# DoS/DDoS (Step 4B)
print("\n3. DoS/DDoS (Step 4B, α=0.10)")
if best_dos:
    for feat, info in best_dos['combo_info'].items():
        direction = rule_config['DoS/DDoS'][feat]
        op = ">=" if direction == 'high' else "<="
        print(f"   IF {feat} {op} {info['value']:.4f}")
    print("   THEN \"DoS/DDoS\"")
    r = best_dos['result']
    print(f"   → Recall={r['Recall']:.4f}, FPR={r['FPR']:.4f}, F1={r['F1']:.4f}")

# Web Attack (Step 4A)
print("\n4. Web Attack (Step 4A, k=2-of-3, α=0.01)")
if best_web:
    print("   IF at least 2 of the following are satisfied:")
    for feat, info in best_web['combo_info'].items():
        direction = rule_config['Web Attack'][feat]
        op = ">=" if direction == 'high' else "<="
        print(f"      - {feat} {op} {info['value']:.4f}")
    print("   THEN \"Web Attack (Suspicious)\"")
    r = best_web['result']
    print(f"   → Recall={r['Recall']:.4f}, FPR={r['FPR']:.4f}, F1={r['F1']:.4f}")


# Performance comparison table
print("\n" + "="*70)
print("Step 3 vs Step 4 Performance Comparison")
print("="*70)
print(f"{'Category':<15} {'Step':>8} {'Recall':>10} {'FPR':>10} {'F1':>10} {'Note':<20}")
print("-"*75)

# Brute Force
s3 = step3_results['Brute Force']
print(f"{'Brute Force':<15} {'Step3':>8} {s3['Recall']:>10.4f} {s3['FPR']:>10.4f} {s3['F1']:>10.4f} {'Unchanged':<20}")

# Botnet
print(f"{'Botnet':<15} {'Step3':>8} {'0.0000':>10} {'0.0440':>10} {'0.0000':>10} {'Failed (for presentation)':<20}")

# DoS/DDoS
s3 = step3_results['DoS/DDoS']
print(f"{'DoS/DDoS':<15} {'Step3':>8} {s3['Recall']:>10.4f} {s3['FPR']:>10.4f} {s3['F1']:>10.4f} {'α=0.05':<20}")
if best_dos:
    r = best_dos['result']
    change = "↑" if r['Recall'] > s3['Recall'] else "↓"
    print(f"{'':<15} {'Step4B':>8} {r['Recall']:>10.4f} {r['FPR']:>10.4f} {r['F1']:>10.4f} {'α=0.10 ' + change:<20}")

# Web Attack
s3 = step3_results['Web Attack']
print(f"{'Web Attack':<15} {'Step3':>8} {s3['Recall']:>10.4f} {s3['FPR']:>10.4f} {s3['F1']:>10.4f} {'AND rule':<20}")
if best_web:
    r = best_web['result']
    change = "↑" if r['Recall'] > s3['Recall'] else "↓"
    print(f"{'':<15} {'Step4A':>8} {r['Recall']:>10.4f} {r['FPR']:>10.4f} {r['F1']:>10.4f} {'k=2-of-3 ' + change:<20}")



Step 4 Final Summary

[Final Rules by Category]
----------------------------------------------------------------------

1. Brute Force (Step 3 unchanged, α=0.01)
   IF Bwd Pkts/s >= 51754.3860
      AND Init Fwd Win Byts >= 26883.0000
   THEN "Brute Force"
   → Recall=0.9660, FPR=0.0088, F1=0.9619

2. Botnet (Step 3, α=0.05) Failed
   IF Flow Duration <= 20.0000
      AND Pkt Len Mean <= 0.0000
   THEN "Botnet"
   → Recall=0.0000, FPR=0.0440 (Detection not possible due to feature limitations)

3. DoS/DDoS (Step 4B, α=0.10)
   IF PSH Flag Cnt >= 1.0000
   IF Init Fwd Win Byts >= 14600.0000
   THEN "DoS/DDoS"
   → Recall=0.5751, FPR=0.0562, F1=0.6958

4. Web Attack (Step 4A, k=2-of-3, α=0.01)
   IF at least 2 of the following are satisfied:
      - Subflow Fwd Byts >= 24782.0440
      - Fwd Header Len >= 400.0000
      - Fwd Pkt Len Std >= 235.6157
   THEN "Web Attack (Suspicious)"
   → Recall=0.2611, FPR=0.0082, F1=0.3767

Step 3 vs Step 4 Performance Comparison
Category            St